In [1]:
import pandas as pd
import numpy as np
import torch
import matplotlib.pyplot as plt
import torch.nn as nn
from torchinfo import summary

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder
from sklearn.impute import KNNImputer
from sklearn.pipeline import Pipeline

In [2]:
rawDataFrame = pd.read_csv('https://raw.githubusercontent.com/gscdit/Breast-Cancer-Detection/refs/heads/master/data.csv')
rawDataFrame

,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Unnamed: 32
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.30010,0.14710,...,17.33,184.60,2019.0,0.16220,0.66560,0.7119,0.2654,0.4601,0.11890,NaN
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.08690,0.07017,...,23.41,158.80,1956.0,0.12380,0.18660,0.2416,0.1860,0.2750,0.08902,NaN
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.19740,0.12790,...,25.53,152.50,1709.0,0.14440,0.42450,0.4504,0.2430,0.3613,0.08758,NaN
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.24140,0.10520,...,26.50,98.87,567.7,0.20980,0.86630,0.6869,0.2575,0.6638,0.17300,NaN
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.19800,0.10430,...,16.67,152.20,1575.0,0.13740,0.20500,0.4000,0.1625,0.2364,0.07678,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
564,926424,M,21.56,22.39,142.00,1479.0,0.11100,0.11590,0.24390,0.13890,...,26.40,166.10,2027.0,0.14100,0.21130,0.4107,0.2216,0.2060,0.07115,NaN
565,926682,M,20.13,28.25,131.20,1261.0,0.09780,0.10340,0.14400,0.09791,...,38.25,155.00,1731.0,0.11660,0.19220,0.3215,0.1628,0.2572,0.06637,NaN
566,926954,M,16.60,28.08,108.30,858.1,0.08455,0.10230,0.09251,0.05302,...,34.12,126.70,1124.0,0.11390,0.30940,0.3403,0.1418,0.2218,0.07820,NaN
567,927241,M,20.60,29.33,140.10,1265.0,0.11780,0.27700,0.35140,0.15200,...,39.42,184.60,1821.0,0.16500,0.86810,0.9387,0.2650,0.4087,0.12400,NaN


In [3]:
rawDataFrame.drop(columns=["id", "Unnamed: 32"], inplace=True)
rawDataFrame.sample(5)

,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,symmetry_mean,...,radius_worst,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst
143,B,12.900,15.92,83.74,512.2,0.08677,0.09509,0.04894,0.03088,0.1778,...,14.480,21.82,97.17,643.8,0.13120,0.2548,0.2090,0.10120,0.3549,0.08118
247,B,12.890,14.11,84.95,512.2,0.08760,0.13460,0.13740,0.03980,0.1596,...,14.390,17.70,105.00,639.1,0.12540,0.5849,0.7727,0.15610,0.2639,0.11780
402,B,12.960,18.29,84.18,525.2,0.07351,0.07899,0.04057,0.01883,0.1874,...,14.130,24.61,96.31,621.9,0.09329,0.2318,0.1604,0.06608,0.3207,0.07247
427,B,10.800,21.98,68.79,359.9,0.08801,0.05743,0.03614,0.01404,0.2016,...,12.760,32.04,83.69,489.5,0.13030,0.1696,0.1927,0.07485,0.2965,0.07662
71,B,8.888,14.64,58.79,244.0,0.09783,0.15310,0.08606,0.02872,0.1902,...,9.733,15.67,62.56,284.4,0.12070,0.2436,0.1434,0.04786,0.2254,0.10840


In [4]:
XTrain, XTest, yTrain, yTest = train_test_split(rawDataFrame.iloc[:,1:], rawDataFrame.iloc[:, 0], stratify=rawDataFrame.iloc[:, 0], test_size=0.2)
XTrain, XTest, yTrain, yTest

(     radius_mean  texture_mean  perimeter_mean  area_mean  smoothness_mean  \
 458        13.00         25.13           82.61      520.2          0.08369   
 379        11.08         18.83           73.30      361.6          0.12160   
 156        17.68         20.74          117.40      963.7          0.11150   
 450        11.87         21.54           76.83      432.0          0.06613   
 100        13.61         24.98           88.05      582.7          0.09488   
 ..           ...           ...             ...        ...              ...   
 127        19.00         18.91          123.40     1138.0          0.08217   
 342        11.06         14.96           71.49      373.9          0.10330   
 407        12.85         21.37           82.63      514.5          0.07551   
 522        11.26         19.83           71.30      388.1          0.08511   
 65         14.78         23.94           97.40      668.3          0.11720   
 
      compactness_mean  concavity_mean  concave po

In [5]:
rawDataFrame["diagnosis"].value_counts(normalize=True)

diagnosis
B    0.627417
M    0.372583
Name: proportion, dtype: float64

In [6]:
yTrain.value_counts(normalize=True)

diagnosis
B    0.626374
M    0.373626
Name: proportion, dtype: float64

In [7]:
encoder = LabelEncoder()
yTrain = encoder.fit_transform(yTrain)
yTest = encoder.fit_transform(yTest)
yTrain, yTest

(array([0, 1, 1, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 1, 1, 0, 0, 0, 0,
        0, 1, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 1, 0,
        0, 1, 1, 1, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0,
        1, 1, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 1, 1, 0, 0, 0,
        1, 0, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 1, 1, 0, 0, 0, 1, 0, 0, 0, 1,
        1, 1, 1, 0, 1, 1, 0, 0, 0, 1, 0, 1, 0, 1, 1, 0, 0, 1, 0, 0, 0, 0,
        0, 1, 1, 0, 0, 1, 1, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        1, 0, 0, 0, 0, 1, 0, 0, 1, 1, 1, 0, 0, 0, 1, 0, 1, 1, 1, 0, 0, 1,
        0, 0, 0, 1, 0, 1, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0,
        1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 1, 1, 0, 0, 0, 0, 1, 0,
        0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0,
        1, 1, 1, 0, 0, 1, 1, 1, 0, 0, 1, 0, 1, 0, 0, 1, 1, 0, 1, 1, 1, 0,
        0, 0, 0, 1, 0, 1, 1, 1, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 1, 0,
        0, 1, 1, 0, 0, 0, 0, 1, 0, 0, 

In [8]:
preprocessPipeline = Pipeline(
    [
        ("Scale", StandardScaler()),
        (
            "Impute",
            KNNImputer(
                n_neighbors=5,
                weights="distance",
                metric="nan_euclidean"
            )
        )
    ]
)

XTrain = preprocessPipeline.fit_transform(XTrain)
XTest = preprocessPipeline.transform(XTest)
XTrain, XTest

(array([[-0.29907546,  1.35995203, -0.36501899, ..., -0.83876043,
         -0.97997834, -1.14719388],
        [-0.84824033, -0.1039957 , -0.751598  , ...,  2.13044276,
          2.05589182,  3.04011004],
        [ 1.03951389,  0.33983607,  1.07956571, ...,  0.57967615,
         -0.7220608 , -0.36427247],
        ...,
        [-0.34197897,  0.48623085, -0.36418853, ..., -0.88794232,
         -0.68099113, -0.14081253],
        [-0.79675612,  0.12837696, -0.83464397, ..., -1.3135194 ,
         -0.56763883, -0.43190563],
        [ 0.21004613,  1.08342857,  0.24910598, ...,  0.73183264,
          0.68745035,  0.27039706]], shape=(455, 30)),
 array([[ 1.58581852, -0.26200911,  1.63182143, ...,  1.70471199,
          1.21642772,  0.43920941],
        [-0.25903219,  0.33983607, -0.22508653, ...,  0.06634013,
         -0.55778211, -0.12836803],
        [ 4.02273761, -0.18764986,  4.03185005, ...,  0.70263089,
         -2.06093211, -1.56164985],
        ...,
        [ 0.93082501, -0.51064785,  0

In [9]:
XTrain = torch.from_numpy(XTrain)
XTrain = torch.tensor(XTrain, dtype = torch.float32)
XTest = torch.from_numpy(XTest)
XTest = torch.tensor(XTest, dtype = torch.float32)
yTrain = torch.from_numpy(yTrain)
yTrain = torch.tensor(yTrain, dtype = torch.float32)
yTest = torch.from_numpy(yTest)
yTest = torch.tensor(yTest, dtype = torch.float32)
XTrain, XTest, yTrain, yTest

/tmp/ipykernel_3242/1706960164.py:2: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  XTrain = torch.tensor(XTrain, dtype = torch.float32)
/tmp/ipykernel_3242/1706960164.py:4: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  XTest = torch.tensor(XTest, dtype = torch.float32)
/tmp/ipykernel_3242/1706960164.py:6: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  yTrain = torch.tensor(yTrain, dtype = torch.float32)
/tmp/ipykernel_3242/1706960164.py:8: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() 

(tensor([[-0.2991,  1.3600, -0.3650,  ..., -0.8388, -0.9800, -1.1472],
         [-0.8482, -0.1040, -0.7516,  ...,  2.1304,  2.0559,  3.0401],
         [ 1.0395,  0.3398,  1.0796,  ...,  0.5797, -0.7221, -0.3643],
         ...,
         [-0.3420,  0.4862, -0.3642,  ..., -0.8879, -0.6810, -0.1408],
         [-0.7968,  0.1284, -0.8346,  ..., -1.3135, -0.5676, -0.4319],
         [ 0.2100,  1.0834,  0.2491,  ...,  0.7318,  0.6875,  0.2704]]),
 tensor([[ 1.5858, -0.2620,  1.6318,  ...,  1.7047,  1.2164,  0.4392],
         [-0.2590,  0.3398, -0.2251,  ...,  0.0663, -0.5578, -0.1284],
         [ 4.0227, -0.1876,  4.0318,  ...,  0.7026, -2.0609, -1.5616],
         ...,
         [ 0.9308, -0.5106,  0.8969,  ...,  1.1053,  0.3868, -0.1614],
         [-0.4535,  0.1423, -0.4555,  ..., -0.5769, -0.3574, -0.3502],
         [ 1.7546,  1.0904,  2.1716,  ...,  2.1059,  4.1685,  0.8401]]),
 tensor([0., 1., 1., 0., 1., 0., 0., 1., 0., 0., 1., 0., 0., 1., 0., 0., 1., 1.,
         0., 0., 0., 0., 0., 1., 1.

In [10]:
XTrain.shape

torch.Size([455, 30])

In [11]:
yTrain = yTrain.reshape(-1, 1)
yTest = yTest.reshape(-1, 1)
yTest[0:20]

tensor([[1.],
        [0.],
        [1.],
        [1.],
        [0.],
        [1.],
        [0.],
        [0.],
        [0.],
        [1.],
        [0.],
        [1.],
        [1.],
        [0.],
        [0.],
        [1.],
        [0.],
        [0.],
        [1.],
        [0.]])

# With Single perceptron

In [12]:
torch.manual_seed(67)
class SingleN():
    def __init__(self, X):
        self.w = torch.rand(X.shape[1], 1, dtype=torch.float32, requires_grad=True)
        self.b = torch.zeros(1, dtype=torch.float32, requires_grad=True)
    
    def forwardPropagation(self, X):
        z = torch.matmul(X, self.w) + self.b
        return torch.sigmoid(z)
    
    def lossFunction(self, y, yHat):
        e = 1e-7
        yHat = yHat.clamp(e, (1 - e))
        loss = -((y * yHat.log()) + ((1 - y) * torch.log(1-yHat)))
        return loss.mean()

In [13]:
epochs = 30_000
alpha = 0.003
model = SingleN(XTrain)

for epoch in range(epochs):

    with torch.no_grad():
        yEvalHat = model.forwardPropagation(XTest)
        lossOnEval = model.lossFunction(yTest, yEvalHat)
        yEvalHat = (yEvalHat > 0.5).int()
        evalAccuracy = (yTest == yEvalHat).float().mean()

    yHat = model.forwardPropagation(XTrain)
    loss = model.lossFunction(yTrain, yHat)

    loss.backward()

    with torch.no_grad():
        model.w -= alpha * model.w.grad
        model.b -= alpha * model.b.grad
    
    model.w.grad.zero_()
    model.b.grad.zero_()

    if((epoch+1) % 3_000 == 0):
        with torch.no_grad():
            yHat = (yHat > 0.5).int()
            trainAccuracy = (yTrain == yHat).float().mean()
            print(f"Epoch: {epoch + 1}, train loss: {loss}, train accuracy: {trainAccuracy*100}% | evaluation loss: {lossOnEval}, evaluation accuracy: {evalAccuracy*100}%")


Epoch: 3000, train loss: 0.09638044983148575, train accuracy: 96.04395294189453% | evaluation loss: 0.06892671436071396, evaluation accuracy: 98.24561309814453%
Epoch: 6000, train loss: 0.07977049052715302, train accuracy: 96.92308044433594% | evaluation loss: 0.06463310122489929, evaluation accuracy: 99.122802734375%
Epoch: 9000, train loss: 0.07497970759868622, train accuracy: 97.80220031738281% | evaluation loss: 0.060649801045656204, evaluation accuracy: 99.122802734375%
Epoch: 12000, train loss: 0.07227292656898499, train accuracy: 97.80220031738281% | evaluation loss: 0.05770944431424141, evaluation accuracy: 99.122802734375%
Epoch: 15000, train loss: 0.0702766552567482, train accuracy: 97.80220031738281% | evaluation loss: 0.055567432194948196, evaluation accuracy: 99.122802734375%
Epoch: 18000, train loss: 0.06866779178380966, train accuracy: 97.80220031738281% | evaluation loss: 0.05394357070326805, evaluation accuracy: 99.122802734375%
Epoch: 21000, train loss: 0.067323923110

# With MLP

In [18]:
class NeuralN:
    def __init__(self, X):

        input_features = X.shape[1]

        # He initialization for ReLU layers
        self.w1 = (
            torch.randn(input_features, 10, dtype=torch.float64)
            * (2 / input_features) ** 0.5
        ).requires_grad_()
        self.b1 = torch.zeros(10, dtype=torch.float64, requires_grad=True)

        self.w2 = (
            torch.randn(10, 7, dtype=torch.float64)
            * (2 / 10) ** 0.5
        ).requires_grad_()
        self.b2 = torch.zeros(7, dtype=torch.float64, requires_grad=True)

        self.w3 = (
            torch.randn(7, 4, dtype=torch.float64)
            * (2 / 7) ** 0.5
        ).requires_grad_()
        self.b3 = torch.zeros(4, dtype=torch.float64, requires_grad=True)

        # Output layer
        self.w4 = (
            torch.randn(4, 1, dtype=torch.float64)
            * (2 / 4) ** 0.5
        ).requires_grad_()
        self.b4 = torch.zeros(1, dtype=torch.float64, requires_grad=True)

    def forwardPropagation(self, X):

        # Layer 1
        z1 = torch.matmul(X, self.w1) + self.b1
        a1 = torch.relu(z1)

        # Layer 2
        z2 = torch.matmul(a1, self.w2) + self.b2
        a2 = torch.relu(z2)

        # Layer 3
        z3 = torch.matmul(a2, self.w3) + self.b3
        a3 = torch.relu(z3)

        # Output layer
        z4 = torch.matmul(a3, self.w4) + self.b4
        yHat = torch.sigmoid(z4)

        return yHat

    def lossFunction(self, y, yHat):

        e = 1e-16

        yHat = yHat.clamp(e, 1 - e)

        loss = (
            -(y * torch.log(yHat))
            - ((1 - y) * torch.log(1 - yHat))
        )

        return loss.mean()

In [25]:
epochs = 20_00_000
alpha = 0.000001

model = NeuralN(XTrain)

for epoch in range(epochs):

    # Clear previous gradients
    if model.w1.grad is not None:
        model.w1.grad.zero_()
        model.b1.grad.zero_()
        model.w2.grad.zero_()
        model.b2.grad.zero_()
        model.w3.grad.zero_()
        model.b3.grad.zero_()
        model.w4.grad.zero_()
        model.b4.grad.zero_()

    # Training forward propagation
    yHat = model.forwardPropagation(XTrain)

    # Training loss
    loss = model.lossFunction(yTrain, yHat)

    # Backpropagation
    loss.backward()

    # Gradient descent
    with torch.no_grad():
        model.w1 -= alpha * model.w1.grad
        model.b1 -= alpha * model.b1.grad

        model.w2 -= alpha * model.w2.grad
        model.b2 -= alpha * model.b2.grad

        model.w3 -= alpha * model.w3.grad
        model.b3 -= alpha * model.b3.grad

        model.w4 -= alpha * model.w4.grad
        model.b4 -= alpha * model.b4.grad

    

    # Logging
    if (epoch + 1) % 1_00_000 == 0:

        # Evaluation
        with torch.no_grad():

            yEvalHat = model.forwardPropagation(XTest)

            lossOnEval = model.lossFunction(
                yTest,
                yEvalHat
            )

            evalPrediction = (yEvalHat > 0.5).int()
            evalAccuracy = (
                (yTest == evalPrediction)
                .float()
                .mean()
            )

        with torch.no_grad():

            trainPrediction = (yHat > 0.5).int()

            trainAccuracy = (
                (yTrain == trainPrediction)
                .float()
                .mean()
            )

            print(
                f"Epoch: {epoch + 1}, "
                f"train loss: {loss.item()}, "
                f"train accuracy: {trainAccuracy.item() * 100:.2f}% | "
                f"evaluation loss: {lossOnEval.item()}, "
                f"evaluation accuracy: {evalAccuracy.item() * 100:.2f}%"
            )
    

Epoch: 100000, train loss: 0.5603276221189725, train accuracy: 68.57% | evaluation loss: 0.5818046794720705, evaluation accuracy: 71.05%
Epoch: 200000, train loss: 0.5037464476333108, train accuracy: 79.12% | evaluation loss: 0.5227746568143364, evaluation accuracy: 79.82%
Epoch: 300000, train loss: 0.46723295408658555, train accuracy: 83.30% | evaluation loss: 0.4844053464861005, evaluation accuracy: 84.21%
Epoch: 400000, train loss: 0.4366520715987178, train accuracy: 87.69% | evaluation loss: 0.4528140292357544, evaluation accuracy: 85.09%
Epoch: 500000, train loss: 0.40939659343802753, train accuracy: 88.13% | evaluation loss: 0.4247075878456076, evaluation accuracy: 85.09%
Epoch: 600000, train loss: 0.3851700659692158, train accuracy: 90.11% | evaluation loss: 0.3993057530122392, evaluation accuracy: 87.72%
Epoch: 700000, train loss: 0.36259863556641675, train accuracy: 90.77% | evaluation loss: 0.37576561732732894, evaluation accuracy: 90.35%
Epoch: 800000, train loss: 0.34094409

In [26]:
print("Class ratio:", yTrain.float().mean().item())

with torch.no_grad():
    yHat = model.forwardPropagation(XTrain)
    yPred = (yHat > 0.5).int()

    print("Predicted positive ratio:", yPred.float().mean().item())
    print("Train loss:", loss.item())
    print("Accuracy:", ((yTrain == yPred).float().mean().item()))

Class ratio: 0.37362638115882874
Predicted positive ratio: 0.35384616255760193
Train loss: 0.17666548621237899
Accuracy: 0.9538461565971375


In [27]:
with torch.no_grad():
    z1 = torch.matmul(XTrain, model.w1) + model.b1
    a1 = torch.relu(z1)

    z2 = torch.matmul(a1, model.w2) + model.b2
    a2 = torch.relu(z2)

    z3 = torch.matmul(a2, model.w3) + model.b3
    a3 = torch.relu(z3)

    z4 = torch.matmul(a3, model.w4) + model.b4
    yHat = torch.sigmoid(z4)

    print("a1 positive:", (a1 > 0).float().mean().item())
    print("a2 positive:", (a2 > 0).float().mean().item())
    print("a3 positive:", (a3 > 0).float().mean().item())

    print("yHat mean:", yHat.mean().item())
    print("yHat min :", yHat.min().item())
    print("yHat max :", yHat.max().item())

    print("b1:", model.b1)
    print("b2:", model.b2)
    print("b3:", model.b3)
    print("b4:", model.b4)

a1 positive: 0.5248351693153381
a2 positive: 0.5729984045028687
a3 positive: 0.6708791255950928
yHat mean: 0.3545401808113231
yHat min : 1.66180966982416e-05
yHat max : 0.997457079224999
b1: tensor([ 0.0268,  0.0471, -0.0454, -0.0475,  0.0307,  0.0093,  0.1074,  0.0138,
        -0.0060,  0.0516], dtype=torch.float64, requires_grad=True)
b2: tensor([-0.0021,  0.0032,  0.1050, -0.0058,  0.0784, -0.0130, -0.0052],
       dtype=torch.float64, requires_grad=True)
b3: tensor([ 0.0383,  0.0407, -0.0157,  0.1220], dtype=torch.float64,
       requires_grad=True)
b4: tensor([0.0709], dtype=torch.float64, requires_grad=True)


In [28]:
with torch.no_grad():
    z1 = XTrain @ model.w1 + model.b1
    a1 = torch.relu(z1)

    z2 = a1 @ model.w2 + model.b2
    a2 = torch.relu(z2)

    z3 = a2 @ model.w3 + model.b3
    a3 = torch.relu(z3)

    z4 = a3 @ model.w4 + model.b4

    print("z4 mean:", z4.mean().item())
    print("z4 std :", z4.std().item())

z4 mean: -1.4825947882063981
z4 std : 2.96555257668311


In [40]:
class NNModel(nn.Module):

    def __init__(self, X):
        super().__init__()
        self.neuralNetwork = nn.Sequential(
            nn.Linear(X.shape[1], 60),
            nn.ReLU(),
            nn.Linear(60, 60),
            nn.ReLU(),
            nn.Linear(60, 1),
            nn.Sigmoid()
        )
    
    def forward(self, X):
        return self.neuralNetwork(X)

nnModel = NNModel(X=XTrain)
summary(nnModel, input_size= XTrain.shape)

Layer (type:depth-idx)                   Output Shape              Param #
NNModel                                  [455, 1]                  --
├─Sequential: 1-1                        [455, 1]                  --
│    └─Linear: 2-1                       [455, 60]                 1,860
│    └─ReLU: 2-2                         [455, 60]                 --
│    └─Linear: 2-3                       [455, 60]                 3,660
│    └─ReLU: 2-4                         [455, 60]                 --
│    └─Linear: 2-5                       [455, 1]                  61
│    └─Sigmoid: 2-6                      [455, 1]                  --
Total params: 5,581
Trainable params: 5,581
Non-trainable params: 0
Total mult-adds (Units.MEGABYTES): 2.54
Input size (MB): 0.05
Forward/backward pass size (MB): 0.44
Params size (MB): 0.02
Estimated Total Size (MB): 0.52

In [41]:
nnModel.parameters()

<generator object Module.parameters at 0x7fd5096ddc40>

## Training Pipeline

In [58]:
print(torch.get_num_threads())
torch.set_num_threads(11)
print(torch.get_num_threads())
epochs = 50_000
alpha = 0.0000001

lossFunc = nn.BCELoss()
optimizer = torch.optim.SGD(nnModel.parameters(), lr=alpha)

for epoch in range(epochs):

    yHat = nnModel(XTrain)
    loss = lossFunc(yHat, yTrain)
    optimizer.zero_grad()

    loss.backward()
    optimizer.step()

    if((epoch +1) % 1_000 == 0):
        with torch.no_grad():

            yEval = nnModel(XTest)
            evalLoss = lossFunc(yEval, yTest)
            yEval = (yEval > 0.5).int()
            evalAccuracy = (yEval == yTest).float().mean()

            yHat = (yHat > 0.5).int()
            trainAccuracy = (yHat == yTrain).float().mean()
        
        print(
                f"Epoch: {epoch + 1}, "
                f"train loss: {loss.item()}, "
                f"train accuracy: {trainAccuracy.item() * 100:.2f}% | "
                f"evaluation loss: {evalLoss.item()}, "
                f"evaluation accuracy: {evalAccuracy.item() * 100:.2f}%"
            )

11
11
Epoch: 1000, train loss: 0.011836285702884197, train accuracy: 99.78% | evaluation loss: 0.03885452821850777, evaluation accuracy: 99.12%
Epoch: 2000, train loss: 0.011836285702884197, train accuracy: 99.78% | evaluation loss: 0.03885452821850777, evaluation accuracy: 99.12%
Epoch: 3000, train loss: 0.011836287565529346, train accuracy: 99.78% | evaluation loss: 0.038854531943798065, evaluation accuracy: 99.12%
Epoch: 4000, train loss: 0.011836287565529346, train accuracy: 99.78% | evaluation loss: 0.03885452821850777, evaluation accuracy: 99.12%
Epoch: 5000, train loss: 0.011836286634206772, train accuracy: 99.78% | evaluation loss: 0.03885452821850777, evaluation accuracy: 99.12%
Epoch: 6000, train loss: 0.011836285702884197, train accuracy: 99.78% | evaluation loss: 0.038854531943798065, evaluation accuracy: 99.12%
Epoch: 7000, train loss: 0.011836285702884197, train accuracy: 99.78% | evaluation loss: 0.038854531943798065, evaluation accuracy: 99.12%
Epoch: 8000, train loss: 